# Visualizing graph eigenvectors as minimum spanning trees

*Converted from `Brouillons, idées.py` (archived source).*

## What is unique here

Graph eigenvectors (especially the Fiedler vector, the 2nd smallest eigenvector of the graph
Laplacian) are usually visualized as a scalar color field on the point cloud. `Brouillons, idées.py`
proposed a complementary view: **draw each eigenvector as a 3D line structure**.

The idea (`VisuEigenVectors`) is:

1. For a given eigenvector, keep only the points where the eigenvector has a **non-zero
   component**;
2. Build the **complete graph** over those points, weighting every edge by the **3D Euclidean
   distance** between the two points;
3. Extract the **minimum spanning tree (MST)** of that complete graph — the set of edges that
   connects all the points with minimum total length;
4. Display the MST edges as an Open3D `LineSet`.

Because the MST uses the shortest edges to span the selected points, it traces the spatial
"backbone" along which the eigenvector varies — a one-dimensional skeleton of the region the
eigenvector activates. This makes eigen-structure visible as geometry rather than color.

## Audience

- **Computer scientists / ML:** the MST is a classical, polynomial-time graph primitive
  (Kruskal 1956; Prim 1957). Applying it to the Euclidean complete graph of the eigenvector's
  support yields a low-cost connected skeleton — an intuitive, dependency-free visualization.
  The commented-out alternative in the original script (remove the two farthest points, then a
  shortest path between them) hints at a spectral "diameter" reading of the eigenvector.
- **Biologists / botanists:** eigenvectors such as the Fiedler vector naturally separate the plant
  along its main axis; drawing them as a tree exposes the branching order of the organs, without
  needing the full segmentation pipeline.


## 1. Imports

We build a point-cloud graph, compute its eigenvectors, and use the library functions
`eigenvector_mst_edges` + `display_eigenvector_mst` added to `display_and_export.py`.


In [ ]:
import os
from pathlib import Path

import numpy as np
import open3d as o3d

from spectral_clustering.similarity_graph import create_connected_riemannian_graph
from spectral_clustering.point_cloud_graph import PointCloudGraph
from spectral_clustering.display_and_export import (
    eigenvector_mst_edges,
    display_eigenvector_mst,
)

os.makedirs("outputs", exist_ok=True)
print("imports ok")


## 2. Build the point-cloud graph and its eigenvectors

The MST visualization only needs the point-cloud graph and its eigenvectors — no segmentation.
We use `compute_graph_eigenvectors` (same call as the pipeline) and read the eigenvector columns
from `G.keigenvec`.


In [ ]:
model_filename = "../data/chenos/cheno_3.ply"
pcd = o3d.io.read_point_cloud(model_filename, format="ply")
SimG, pcdfinal = create_connected_riemannian_graph(point_cloud=pcd, method="knn", nearest_neighbors=18)
G = PointCloudGraph(SimG)
G.pcd = pcdfinal

G.compute_graph_eigenvectors(is_sparse=True, k=6, smallest_first=True, laplacian_type="classical")
print("eigenvectors shape:", G.keigenvec.shape)


## 3. Compute the eigenvector MSTs

`eigenvector_mst_edges` returns, for each eigenvector, the list of MST edges over the points
with a non-zero component, weighted by 3D Euclidean distance.


In [ ]:
mst_edges = eigenvector_mst_edges(G.keigenvec, G, k=G.keigenvec.shape[1])
for j, edges in mst_edges.items():
    print(f"eigenvector {j}: {len(edges)} MST edges")


## 4. Display an eigenvector MST

`display_eigenvector_mst` wraps the MST edges into an Open3D `LineSet` and draws it on the
point cloud. In a headless/notebook context the interactive `draw_geometries` window opens;
set ``eigenvector_index=1`` (the Fiedler vector) or any other index to inspect it.


In [ ]:
# display_eigenvector_mst(pcd, mst_edges, eigenvector_index=1)
print("Interactive display skipped in this run; set eigenvector_index to inspect.")


## 5. Interpretation

Each eigenvector MST is a connected, minimum-length set of segments that spans the points
where that eigenvector is active. Interpreting the results:

- **Fiedler vector (index 1)** — its MST follows the longest, most central structure of the
  plant (typically the main stem), because the eigenvector changes sign across the two halves
  the Laplacian most strongly separates.
- **Higher eigenvectors** — their MSTs highlight secondary structures (limbs, branches) and
  their branching order.
- The MST is a **skeleton**: it reveals *where* the eigenvector lives and its 1-D organization,
  complementing the usual scalar color map.

## Summary

- **Library additions** (preserving the archived draft's unique idea):
  - `display_and_export.eigenvector_mst_edges`
  - `display_and_export.display_eigenvector_mst`
- The draft's complete-graph MST is O(n^2) and infeasible at plant scale; the library version
  builds the MST over the point-cloud graph's own edges (O(E log V)), keeping the same skeleton
  intent. The rest of `Brouillons, idées.py` (region growing, small-cluster removal, topological
  energy) is already implemented in `quotient_graph.py`, `quotientgraph_operations.py` and
  `topological_energy.py`, and is not duplicated here.
